In [145]:
import json
from PIL import Image, ImageDraw, ImageFont
import subprocess

In [146]:
PROJECT_FOLDER = '/Users/sangdo/Downloads/movie-top-list/'
TRAILER_FOLDER = PROJECT_FOLDER + 'trailers/'
BG_FILE = PROJECT_FOLDER + 'timeline_template/timeline_30_items.png'
OUTPUT_LONG_VIDEO_FOLDER = PROJECT_FOLDER + 'long_video/'

first_video_x = 13
firxt_video_y = 65
CANVAS_WIDTH = 19980    #length of background image
CANVAS_HEIGHT = 1080
VIEWPORT_WIDTH = 1920
VIEWPORT_HEIGHT = 1080
VIDEO_DURATION = 480
CLIP_W = 640
CLIP_H = 360
#Scrollable distance: 19980 - 1920 = 18060 px
#Desired duration: 480
#Required speed: 18060 / 480 = 37.625 px/sec
SCROLL_SPEED = ((CANVAS_WIDTH - VIEWPORT_WIDTH) / VIDEO_DURATION) * 1  #pixels per sec

<h2>Find list of trailers of the actor<h2>

In [147]:
import pymongo
import os
import subprocess

from dotenv import load_dotenv
load_dotenv(override=True) 

db_client = pymongo.MongoClient(os.environ['REMOTE_MONGO_DB'])
db = db_client['db_movie']   


In [148]:
def get_trailer_ids(actor_name):
    ids = []
    tb_paycheck = db['tb_paycheck']
    data = tb_paycheck.find({'actor': actor_name}).sort({'year':1}) #.limit(4)

    for item in data:
        ids.append(item['trailer'])
    return ids

#
# get_trailer_ids('Tom Cruise')

<h2>Put a video at position X, Y inside a video</h2>

In [ ]:
def put_video_inside_video(actor_key, trailer_ids):
    inputs = ""
    filters = ""

    # build ffmpeg input list
    i = 0
    for trailer_id in trailer_ids:
        filepath = TRAILER_FOLDER + actor_key + "/"+trailer_id+".mp4"
        inputs += f"-i {filepath} "

        x = first_video_x + i * CLIP_W  #position of the clip
        trigger_x = VIEWPORT_WIDTH * 2 / 3    #start playing while reach to the center

        delay = max((x - trigger_x) / SCROLL_SPEED, 0)
        play_time = 20

        filters += (
            f"[{i+1}:v]"
            f"scale={CLIP_W}:{CLIP_H},"
            f"trim=duration={play_time},"
            f"tpad=start_duration={delay}:start_mode=clone,"
            f"tpad=stop_duration=500:stop_mode=clone"
            f"[v{i}];"
        )

        i = i + 1

    # create base canvas
    filters += f"[0:v]scale={CANVAS_WIDTH}:{CANVAS_HEIGHT}[base];"

    last = "base"

    # overlay clips in position
    i = 0
    for trailer_id in trailer_ids:
        x = 666 * i + first_video_x
        filters += f"[{last}][v{i}]overlay={x}:{firxt_video_y}:eof_action=pass[tmp{i}];"
        last = f"tmp{i}"
        i = i + 1

    # scrolling camera
    filters += f"[{last}]crop={VIEWPORT_WIDTH}:{VIEWPORT_HEIGHT}:x='t*{SCROLL_SPEED}':y=0[out]"  #standard youtube landscape video size
    output_filename = OUTPUT_LONG_VIDEO_FOLDER + actor_key + ".mp4"
    cmd = f'ffmpeg -y -loop 1 -i {BG_FILE} {inputs} -filter_complex "{filters}" -map "[out]" -t 480 -r 30 -c:v libx264 -preset ultrafast -crf 23 -movflags +faststart -pix_fmt yuv420p {output_filename}'
    print("Running FFmpeg...")
    subprocess.run(cmd, shell=True)

#test
actor_name = 'Tom Cruise'
trailer_ids = get_trailer_ids(actor_name)
print(trailer_ids)
put_video_inside_video('tom_cruise', trailer_ids)    #size w 19980: 8 minutes ~ 

['iIaAL7JTEgE', 'ecBD3dRnho8', 'sCmYN6TLd8A', 'L8Pbjh4EZRk', 'KUd3gwaf0KQ', 'xgVo96JaqeM', 'KnamcFv_N9Q', 'hSPtsCQq52k', 'k09OX40NLUw', 'lG7DGMgfOb8', 'T50_qHEOahQ', 'MPr0OyV9_c0', 'jaasIlkad1Q', '4oVva0muTE8', '0n02lrQ_5Vo', 'Omgf9WuGwq8', '-9LyQxKVJO4', 'JGPl86DBNNs', 'EDGYVFZxsXQ', 'jbvzQLMqfH0', 'Q-oxhxD32MM', 'XmIIgE7eSak', 'vw61gCe2oqI', 'gOW_azQbOjw', 'aRwrdbcAh2s', 'IjHgzkQM2Sg', 'AEBIJRAkujM', 'giXco2jaZ_4', 'avz06PDqDbM', 'fsQgc9pCyDU']
Running FFmpeg...


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, png_pipe, from '/Users/sangdo/Downloads/movie-top-list/timeline_template/timeline_30_items.png':
  Duration: N/A, bitrate: N/A
  Stream #0:0: Video: png, rgba(pc, gbr/unknown/unknown), 19980x10